> **3.1 Task 1: Bijection Learning**
> 
> Each sequence is derived from a new random bijection $\pi : \{1, \dots, V\} \to \{1, \dots, V\}$ with $V = 20$. At position $k$, the model has observed $k - 1$ distinct input–output pairs and must predict $\pi(x_k)$. Because inputs never repeat, the Bayes-optimal posterior over $\pi(x_k)$ is uniform over the $V - k + 1$ unseen values.
> 
> **Bayesian ground truth.** Let $O_{k-1}$ be observed outputs. Then
> 
> $$p(\pi(x_k) = y \mid \text{context}) = \begin{cases} \frac{1}{V-k+1}, & y \notin O_{k-1}, \\ 0, & y \in O_{k-1}, \end{cases}$$
> 
> with entropy $H_{\text{Bayes}}(k) = \log_2(V - k + 1)$.
> 
> **Evaluation.** We compute MAE over a held-out set of 2,000 bijections. Because $20! \approx 2.4 \times 10^{18}$ possible bijections exist and training uses only $10^5$ samples, no bijection is seen twice; the task enforces true hypothesis elimination.
> 
> **Sequence format.** Each training example is tokenized as
> 
> $$[x_1, y_1, \text{SEP}, x_2, y_2, \text{SEP}, \dots, x_{19}, \text{SEP}],$$
> 
> with teacher forcing at every $y_k$ position.


Let's build the dataset.

In [9]:
import torch
from torch.utils.data import Dataset
import numpy as np

class BijectionTask1(Dataset):
    """
    Task 1 from Section 3.1: Pure Hypothesis Elimination.
    Inputs never repeat. The entropy staircase is constant for all samples.
    """
    def __init__(self, num_samples, V=20):
        self.num_samples = num_samples
        self.V = V
        self.sep_token = 0 
        
        # Since inputs never repeat in Task 1, the Bayesian Ground Truth 
        # is the same 'staircase' for every sequence. 
        # We calculate it once here.
        self.staircase_entropy = [np.log2(V - k) for k in range(V - 1)]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 1. Create a fresh secret mapping (The Bijection)
        mapping = np.random.permutation(np.arange(1, self.V + 1))
        bijection = {i+1: val for i, val in enumerate(mapping)}
        
        # 2. Randomize the order of inputs (x1, x2, ... x19)
        # Note: We only go to 19 because the 20th is deterministic
        x_order = np.random.permutation(np.arange(1, self.V + 1))
        
        # 3. Build sequence: [x1, y1, SEP, x2, y2, SEP, ... x19, y19, SEP]
        input_ids = []
        for i in range(self.V - 1):
            xk = x_order[i]
            yk = bijection[xk]
            input_ids.extend([xk, yk, self.sep_token])
            
        return {
            "input_ids": torch.tensor(input_ids[:-1], dtype=torch.long), # The sequence
            "labels": torch.tensor(input_ids[1:], dtype=torch.long),    # Targets for next-token prediction
            "staircase": torch.tensor(self.staircase_entropy, dtype=torch.float)
        }

# --- Verification ---
task1_data = BijectionTask1(num_samples=10)
first_sample = task1_data[0]

print("Sequence structure:", first_sample['input_ids'].tolist())
print("Staircase Entropy (bits per step):", [round(e, 3) for e in first_sample['staircase'].tolist()])

# --- Dataset Shape Analysis ---
print(f"Number of samples in dataset: {len(task1_data)}")
print(f"Length of a single sequence (L): {first_sample['input_ids'].shape[0]} tokens")
print(f"Number of prediction steps (k): {first_sample['staircase'].shape[0]} steps")

Sequence structure: [8, 18, 0, 17, 7, 0, 3, 4, 0, 6, 2, 0, 11, 17, 0, 13, 5, 0, 15, 15, 0, 1, 12, 0, 20, 8, 0, 7, 13, 0, 10, 9, 0, 2, 10, 0, 12, 14, 0, 14, 3, 0, 4, 16, 0, 18, 11, 0, 19, 19, 0, 5, 6, 0, 9, 1]
Staircase Entropy (bits per step): [4.322, 4.248, 4.17, 4.087, 4.0, 3.907, 3.807, 3.7, 3.585, 3.459, 3.322, 3.17, 3.0, 2.807, 2.585, 2.322, 2.0, 1.585, 1.0]
Number of samples in dataset: 10
Length of a single sequence (L): 56 tokens
Number of prediction steps (k): 19 steps


### Dataset Dimensions & Metadata

1.  **`input_ids` shape: `[56]`**
    *   **Why 56?** We have 19 pairs. Each pair is 3 tokens (`x`, `y`, `SEP`). $19 \times 3 = 57$. We remove the last token (`SEP`) because there is nothing to predict after it. So, $57 - 1 = 56$.
2.  **`labels` shape: `[56]`**
    *   This is the same length as `input_ids`. It is shifted right by 1, so the model learns to predict the "next" token at every position.
3.  **`staircase` shape: `[19]`**
    *   This contains the 19 levels of uncertainty. 
    *   **Mapping:** The model's entropy at `input_ids` indices **0, 3, 6, 9...** (the positions of $x_1, x_2, x_3...$) should match these 19 values.